# Neuro-Symbolic LLM — complete JupyterHub benchmark

**Upload this file to JupyterHub, choose a Python 3.11 or 3.12 kernel on an NVIDIA GPU server, then run the cells from top to bottom. All installation and training run on JupyterHub.**

This notebook clones `iCog-Labs-Dev/Neuro-Symbolic-LLM` at `benchmark-final` (or a PR you select), installs the benchmark dependencies in the active kernel, trains A1/A2/A3, evaluates saved checkpoints, displays **internal and external metric tables and every PNG visualization**, and creates a results ZIP.

The default is the **headline research profile**: three seeds, rank 16, A3 K in [1, 2, 4, 8], sequential and joint training, and the repository's configured epochs (currently three). This trains on **real LexGLUE text** from EURLEX, LEDGAR and SCOTUS. The pinned configuration uses 5,000/20,000/500 training examples respectively, with separate validation and test splits. These are real-data subsets, not synthetic examples. Choose `full` to add rank, held-out and commutator sweeps; it can be very expensive. Dataset and model downloads require Internet access on the server.

Evaluation computes measurements from the trained checkpoints. Metrics needing independent labels or interventions remain **N/A** unless their required evidence is supplied; next-token accuracy is not task-classification accuracy.


In [ ]:
import os
import re
import sys
from pathlib import Path

REPOSITORY = "https://github.com/iCog-Labs-Dev/Neuro-Symbolic-LLM.git"
GIT_REF = "benchmark-final"
PR_NUMBER = None  # Optional: set to the GitHub PR number, e.g. 42.
USE_GPU = True  # False only for an intentional CPU run.
PROFILE = "headline"  # Research default; "full" adds rank/protocol sweeps.
EPOCHS = None  # Use the repository's configured training epochs.
RUN_NAME = "headline-real-data"  # Use a NEW name when changing experiment settings.
REPORT_RANK = 16
RUN_TESTS = True

# Files remain under the directory where this notebook is running.
NOTEBOOK_DIR = Path.cwd().resolve()
WORK_DIR = NOTEBOOK_DIR / "benchmark-work"
REPO_DIR = WORK_DIR / "Neuro-Symbolic-LLM"
OUTPUT_DIR = REPO_DIR / "runs" / RUN_NAME
REPORT_DIR = REPO_DIR / "report_and_visualization"
assert PROFILE in {"smoke", "headline", "full"}
assert re.fullmatch(r"[A-Za-z0-9_-]+", RUN_NAME), (
    "Use letters, numbers, _ or - in RUN_NAME."
)
assert EPOCHS is None or isinstance(EPOCHS, int) and EPOCHS > 0
assert sys.version_info >= (3, 11), "Choose a Python 3.11+ kernel."
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Configure the child training processes before they import JAX.
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["MPLBACKEND"] = "Agg"
os.environ["JAX_PLATFORMS"] = "cuda" if USE_GPU else "cpu"
print("JupyterHub Python:", sys.executable)
print("Repository:", REPO_DIR)
print("Results:", OUTPUT_DIR)

## 1. Initialize GitHub checkout

This uses Git from the JupyterHub server. It fetches the selected branch or PR and records its exact commit. If the repository is private, configure Git authentication in the JupyterHub terminal first; do not paste a token into this notebook. Existing local source edits cause a stop so they are preserved. An existing run resumes its verified checkpoints only if its source and configuration match.


In [ ]:
import subprocess


def run_command(command, cwd=None):
    print("Running:", " ".join(map(str, command)), flush=True)
    subprocess.run(list(map(str, command)), cwd=cwd, check=True)


if not (REPO_DIR / ".git").is_dir():
    if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a Git checkout. Choose another WORK_DIR."
        )
    run_command(
        ["git", "clone", "--branch", GIT_REF, "--single-branch", REPOSITORY, REPO_DIR]
    )
else:
    dirty = subprocess.check_output(
        ["git", "status", "--porcelain", "--untracked-files=normal"],
        cwd=REPO_DIR,
        text=True,
    ).strip()
    if dirty:
        raise RuntimeError(
            "Existing checkout has edits. Commit/preserve them or choose a new WORK_DIR.\n"
            + dirty
        )

fetch_ref = f"refs/pull/{int(PR_NUMBER)}/head" if PR_NUMBER is not None else GIT_REF
run_command(["git", "fetch", "origin", fetch_ref], cwd=REPO_DIR)
run_command(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR)
COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()
NEW_RUNNER = REPO_DIR / "benchmark" / "run_benchmark.py"
CONSOLIDATED = NEW_RUNNER.is_file()
assert CONSOLIDATED or (REPO_DIR / "experiments" / "run_benchmark.py").is_file(), (
    "Selected ref does not contain the benchmark."
)
print("Commit:", COMMIT)
print(
    "Runner:",
    "consolidated single file"
    if CONSOLIDATED
    else "legacy layout (handled automatically)",
)

## 2. Install dependencies on JupyterHub

Run this cell on the web server. It uses **this notebook's Python kernel**, reads the checked-out dependency pins (including FabricPC and TorchAX), and installs the model, metric, plotting and test dependencies. The parser, web service and queue dependencies are not needed for this benchmark. NVIDIA GPU mode installs the repository's CUDA 12 JAX extra; the host must already provide a compatible NVIDIA driver.

If Jupyter asks to restart after installation, restart the kernel, rerun the configuration and checkout cells, then continue with the GPU check. You do not need to rerun this installation cell after a restart. A failed dependency download should be resolved before continuing; do not silently remove pinned packages.


In [ ]:
import tomllib

project = tomllib.loads((REPO_DIR / "pyproject.toml").read_text(encoding="utf-8"))[
    "project"
]
wanted = {
    "fabricpc",
    "jax",
    "jaxlib",
    "optax",
    "orbax-checkpoint",
    "torch",
    "transformers",
    "accelerate",
    "datasets",
    "sentencepiece",
    "protobuf",
    "torchax",
    "numpy",
    "pyyaml",
    "tqdm",
}
dependencies = []
for requirement in project["dependencies"]:
    name = re.match(r"[A-Za-z0-9_.-]+", requirement).group().lower().replace("_", "-")
    if name in wanted and not (USE_GPU and name in {"jax", "jaxlib"}):
        dependencies.append(requirement)
dependencies += project["optional-dependencies"]["visualization"]
if USE_GPU:
    dependencies += project["optional-dependencies"]["gpu"]
dependencies += ["pytest>=8", "pytest-asyncio>=0.23"]

run_command([sys.executable, "-m", "pip", "install", *dependencies])
run_command([sys.executable, "-m", "pip", "install", "--no-deps", "-e", REPO_DIR])
print(
    "JupyterHub dependency installation finished. Continue with the subprocess GPU check below."
)

## 3. Check GPU and training imports

This uses a fresh Python process, so it sees the newly installed packages without importing JAX into the notebook kernel. GPU mode stops if JAX cannot see a CUDA device. Use your JupyterHub GPU server/profile when starting the session; a browser alone does not supply a GPU.


In [ ]:
probe = """
import importlib.metadata as metadata
import jax
import torch
import torchax
from frozenllm.substrate import FrozenSubstrate
from stages.stage_A.a3 import A3HybridConfig, A3Model
print('JAX devices:', jax.devices())
print('Torch CUDA available:', torch.cuda.is_available())
for name in ['jax', 'jaxlib', 'torch', 'torchax', 'fabricpc', 'transformers', 'datasets', 'matplotlib', 'pandas']:
    print(name, metadata.version(name))
"""
if USE_GPU:
    probe += "\nassert any(device.platform == 'gpu' for device in jax.devices()), 'No JAX CUDA GPU found. Select a GPU server or set USE_GPU=False intentionally.'\n"
run_command([sys.executable, "-c", probe], cwd=REPO_DIR)
if USE_GPU:
    try:
        run_command(["nvidia-smi"], cwd=REPO_DIR)
    except FileNotFoundError:
        print(
            "nvidia-smi is unavailable; JAX device detection above is the required check."
        )

## 4. Verify the benchmark tests and plan the experiment

The consolidated version has one test file. The legacy version runs its existing benchmark test directory. Planning does not train models. Inspect the run count before starting a large profile. To change settings for an existing experiment, choose a new `RUN_NAME`; the existing manifest is the source of truth when resuming.


In [ ]:
import json

if RUN_TESTS:
    test_target = (
        "tests/benchmark/test_benchmark.py" if CONSOLIDATED else "tests/benchmark"
    )
    run_command(
        [sys.executable, "-m", "pytest", test_target, "-q", "--tb=short"], cwd=REPO_DIR
    )

RUNNER = (
    [sys.executable, str(NEW_RUNNER)]
    if CONSOLIDATED
    else [sys.executable, "-m", "experiments.run_benchmark"]
)
manifest_path = OUTPUT_DIR / "manifest.json"
if not manifest_path.is_file():
    plan = RUNNER + ["plan", "--profile", PROFILE, "--output-dir", str(OUTPUT_DIR)]
    if EPOCHS is not None:
        plan += ["--epochs", str(EPOCHS)]
    run_command(plan, cwd=REPO_DIR)
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print("Planned runs:", len(manifest["runs"]))
print("Profile:", manifest.get("profile"))
print("Methods:", manifest.get("methods"))
print("Seeds:", manifest.get("seeds"))
print("Ranks:", manifest.get("rank_values"))
print("A3 K values:", manifest.get("k_values"))
print("Epochs:", manifest.get("epochs"))
print("Source commit:", manifest.get("source", {}).get("git_commit"))

## 5. Train → evaluate metrics → generate all tables and visualizations

Run this cell and keep the JupyterHub server alive. It displays progress and recent training log lines every 30 seconds. The first run downloads GPT-2 and the pinned legal-domain datasets. Re-running this cell resumes verified completed runs. Interruption stops the active stage; previously completed checkpoints remain on disk.

The consolidated runner executes the entire pipeline in one command. The fallback automatically invokes the older train/eval/report commands if the selected remote branch has not received the consolidation yet. Evaluation must finish before report generation. An exception includes the log tail and stops the notebook.


In [ ]:
import time


def run_with_progress(command):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    console_log = OUTPUT_DIR / "notebook-console.log"
    print("Running:", " ".join(map(str, command)), flush=True)
    started = time.monotonic()
    with console_log.open("w", encoding="utf-8") as stream:
        process = subprocess.Popen(
            list(map(str, command)),
            cwd=REPO_DIR,
            stdout=stream,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
        try:
            while process.poll() is None:
                time.sleep(5)
                if time.monotonic() - started < 5:
                    continue
                if int(time.monotonic() - started) % 30 < 5:
                    state = json.loads(manifest_path.read_text(encoding="utf-8"))
                    counts = {}
                    for run in state.get("runs", []):
                        status = run.get("status", "pending")
                        counts[status] = counts.get(status, 0) + 1
                    print(
                        f"Elapsed {(time.monotonic() - started) / 60:.1f} min | {counts}",
                        flush=True,
                    )
                    logs = (
                        sorted(
                            (OUTPUT_DIR / "logs").glob("*.log"),
                            key=lambda path: path.stat().st_mtime,
                        )
                        if (OUTPUT_DIR / "logs").is_dir()
                        else []
                    )
                    active_log = logs[-1] if logs else console_log
                    tail = active_log.read_text(
                        encoding="utf-8", errors="replace"
                    ).splitlines()[-3:]
                    if tail:
                        print("\n".join(tail), flush=True)
        except KeyboardInterrupt:
            import signal

            os.killpg(process.pid, signal.SIGTERM)
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
            print("Stopped active training. Re-run to resume verified completed runs.")
            raise
    output = console_log.read_text(encoding="utf-8", errors="replace")
    print("\n".join(output.splitlines()[-60:]))
    if process.returncode:
        raise RuntimeError(
            f"Benchmark failed (exit {process.returncode}). Full log: {console_log}"
        )


if CONSOLIDATED:
    command = RUNNER + [
        "run",
        "--output-dir",
        str(OUTPUT_DIR),
        "--report-rank",
        str(REPORT_RANK),
        "--visualization-dir",
        str(REPORT_DIR),
    ]
    run_with_progress(command)
else:
    run_with_progress(RUNNER + ["train", "--output-dir", str(OUTPUT_DIR)])
    run_with_progress(RUNNER + ["eval", "--output-dir", str(OUTPUT_DIR)])
    run_with_progress(
        RUNNER
        + [
            "report",
            "--output-dir",
            str(OUTPUT_DIR),
            "--report-rank",
            str(REPORT_RANK),
            "--visualization-dir",
            str(REPORT_DIR),
        ]
    )

evaluation = json.loads((OUTPUT_DIR / "evaluation.json").read_text(encoding="utf-8"))
print(
    f"Completed valid runs: {evaluation['completed_runs']}/{evaluation['planned_runs']}"
)
if evaluation["completed_runs"] != evaluation["planned_runs"]:
    print(
        "Some runs are missing/invalid. Review coverage and reasons in the report before comparing methods."
    )

## 6. Display comparison tables and plots

Both metric families are shown below. Use the fold-out sections to inspect every generated image. Images are embedded directly, so JupyterHub does not need to serve files from a separate local web server. N/A means the required measurement is missing, not zero.


In [ ]:
import base64
import html

from IPython.display import HTML, Markdown, display


def show_report(folder, title):
    reports = sorted(folder.glob("*.md"))
    if not reports:
        raise FileNotFoundError(f"No generated report found in {folder}")
    display(Markdown(f"# {title}"))
    # Tables/prose render here; image paths are rendered as embedded PNGs below.
    text = reports[0].read_text(encoding="utf-8")
    display(
        Markdown(
            "\n".join(
                line for line in text.splitlines() if not line.strip().startswith("![")
            )
        )
    )
    figures = sorted((folder / "figures").glob("*.png"))
    gallery = []
    for path in figures:
        encoded = base64.b64encode(path.read_bytes()).decode("ascii")
        gallery.append(
            f'<details open><summary>{html.escape(path.stem)}</summary><img style="max-width:100%;height:auto" src="data:image/png;base64,{encoded}" alt="{html.escape(path.stem)}"></details>'
        )
    display(HTML("".join(gallery)))
    print(f"{len(figures)} figures displayed for {title}.")


show_report(
    REPORT_DIR / "stage_A" / "comparison",
    "A1 / A2 / A3 comparison — internal and external metrics",
)

In [ ]:
# All method-specific tables, learning curves and diagnostics.
for method in ("A1", "A2", "A3"):
    show_report(
        REPORT_DIR / "stage_A" / method,
        f"{method} — training, internal metrics and external metrics",
    )

## 7. Download the report, measurements and logs

The ZIP includes reports, images, evaluation, manifest, materialized configs and training logs. Checkpoints are excluded by default to keep the download small. Set `INCLUDE_CHECKPOINTS = True` if you also want saved adapter weights. Model and Hugging Face caches are not included. The environment package versions and exact Git commit are included for reproducibility.


In [ ]:
import zipfile

from IPython.display import FileLink

INCLUDE_CHECKPOINTS = False
archive = NOTEBOOK_DIR / f"benchmark-results-{RUN_NAME}.zip"
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for directory, prefix in (
        (OUTPUT_DIR, "run"),
        (REPORT_DIR, "report_and_visualization"),
    ):
        for path in sorted(directory.rglob("*")):
            if path.is_file() and (INCLUDE_CHECKPOINTS or path.suffix != ".npz"):
                bundle.write(path, f"{prefix}/{path.relative_to(directory).as_posix()}")
    bundle.writestr("environment/pip-freeze.txt", freeze)
    bundle.writestr("environment/commit.txt", COMMIT + "\n")
    bundle.writestr(
        "environment/run-settings.json",
        json.dumps(
            {
                "repository": REPOSITORY,
                "ref": GIT_REF,
                "pr": PR_NUMBER,
                "commit": COMMIT,
                "profile": PROFILE,
                "epochs": EPOCHS,
                "report_rank": REPORT_RANK,
            },
            indent=2,
        ),
    )
print("Download your complete benchmark output:")
display(FileLink(archive.name))

## Regenerate figures later without training again

Re-run the configuration and checkout cells, then the following optional cell. It reads the saved evaluation and regenerates tables and images. If you add audit measurements under `runs/<RUN_NAME>/measurement_inputs/`, rerun **evaluation** first and then regenerate the report. Source code should remain at the same commit as the planned experiment.

For larger experiments, start a new `RUN_NAME` and set `PROFILE = "headline"` (multi-seed, K sweep, sequential/joint) or `"full"` (rank and protocol sweeps). The selected A3 K is chosen from validation loss, not test loss. Some internal research metrics require independent evidence described in `docs/benchmark_metrics.md`.


In [ ]:
# Optional: uncomment to regenerate reports without retraining.
# run_command(RUNNER + ["report", "--output-dir", str(OUTPUT_DIR), "--report-rank", str(REPORT_RANK), "--visualization-dir", str(REPORT_DIR)], cwd=REPO_DIR)
